In [ ]:
import kaggle_benchmarks as kbench

import json
import re
import math
from datetime import datetime

def extract_json(text):
    if not text: return None
    fence = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if fence:
        blob = fence.group(1)
    else:
        start, end = text.find("{"), text.rfind("}")
        if start == -1 or end == -1 or end <= start: return None
        blob = text[start:end + 1]
    try: return json.loads(blob)
    except: return None

def numeric_pass(answer_text, ground_truth, rel_tol=0.015):
    def parse_physics_number(text):
        s = str(text).replace(",", "").strip().lower()
        s = re.sub(r"\\times\s*10\s*(\^|e)\s*{{?(-?\d+)}}?", r"e\2", s)
        s = re.sub(r"\*\s*10\s*(\^|e)\s*{{?(-?\d+)}}?", r"e\2", s)
        match = re.search(r"[-+]?\d*\.?\d+(?:[eE^][-+]?\d+)?", s)
        if match:
            try: return float(match.group(0).replace("^", "e"))
            except: return None
        return None
    
    pred = parse_physics_number(answer_text)
    try:
        target = float(ground_truth)
        if pred is None: return False
        if target == 0: return abs(pred) < 1e-9
        return math.isclose(pred, target, rel_tol=rel_tol)
    except: return False

def build_trace(*, task_id, llm, prompt, response, parsed, final_answer, passed, failure_mode):
    return {
        "timestamp_utc": datetime.utcnow().isoformat() + "Z",
        "task_id": task_id,
        "model": str(llm),
        "pass": bool(passed),
        "failure_mode": failure_mode,
        "final_answer": final_answer,
        "raw_output": response,
        "parsed_output": parsed,
        "prompt": prompt
    }

# ----------------------------
# Task 39: Synthetic Gravity Impulse
# ----------------------------
TASK_ID = "fp_39"
GROUND_TRUTH = 0.798

@kbench.task(name="FP-39 Synthetic Gravity Impulse", description="Physics")
def task_39(llm) -> tuple[int, int]:
    prompt = """You are solving a frontier physics problem. Return valid JSON only.\n\nSynthetic gravity G=12.0. m1=80kg (fixed), m2=1.60kg (on x-axis with spring k=2.0). m3=0.8kg passes along x=0 with v0=0.4m/s. d=0.3m. What is total oscillation energy E_osc in Joules (3 s.f.) long after m3 passes?\n\nReturn JSON: {\"final_answer\": \"<value>\"}"""
    response = llm.prompt(prompt)
    parsed = extract_json(response)
    final_ans = parsed.get("final_answer", "") if parsed else ""
    passed = numeric_pass(final_ans, GROUND_TRUTH)
    return (1 if passed else 0, 1)


In [ ]:
task_39.run(kbench.llm)